# Chapter 7 — Breaking the Detector

**Book alignment:** Hallucination From First Principles, Chapter 7

**Question this notebook isolates:** Do relation-reversal, support-deletion, and rank-change move the Chapter 5 metric as the attack taxonomy predicts (content preserves the verdict, context and configuration flip it)?

Synthetic embeddings below demonstrate the geometry type only and do not reproduce the book's empirical runs.

In [ ]:
import numpy as np

SEED = 7
rng = np.random.default_rng(SEED)
TAU = 0.20  # frozen containment threshold: H <= TAU -> ACCEPT
print("numpy", np.__version__, "seed", SEED, "tau", TAU)

## 1 — Content attack: relation reversal preserves the verdict

A role/binding swap changes truth while preserving topic material. Synthetic analogue: the same in-span direction plus tiny noise. The oracle label flips to UNSUPPORTED but containment should barely move, so the ACCEPT verdict is preserved (false acceptance).

In [ ]:
def hallucination_energy(claim_vec, evidence_vecs, rank_r=8):
    c = np.asarray(claim_vec, dtype=np.float32)
    E = np.asarray(evidence_vecs, dtype=np.float32)
    if E.ndim == 1:
        E = E.reshape(1, -1)
    c = c / max(np.linalg.norm(c), 1e-12)
    E = E / np.where(np.linalg.norm(E, axis=1, keepdims=True) < 1e-12, 1.0,
                     np.linalg.norm(E, axis=1, keepdims=True))
    _, s, Vt = np.linalg.svd(E, full_matrices=False)
    basis = Vt[:min(rank_r, Vt.shape[0])].T
    coords = basis.T @ c
    return float(np.clip(1.0 - float(coords @ coords), 0.0, 1.0))


D = 32
E = np.zeros((4, D), dtype=np.float32)
E[0, 0] = E[1, 1] = E[2, 2] = E[3, 3] = 1.0
c_good = np.zeros(D, dtype=np.float32)
c_good[0], c_good[1], c_good[2], c_good[3] = 0.6, 0.5, 0.4, 0.3
# Oracle-UNSUPPORTED reversal: identical material, perturbed binding proxy.
c_rev = c_good + 0.005 * rng.standard_normal(D).astype(np.float32)
h_good = hallucination_energy(c_good, E, 4)
h_rev = hallucination_energy(c_rev, E, 4)
print(f"H(supported)={h_good:.4f} -> {'ACCEPT' if h_good <= TAU else 'REJECT'}")
print(f"H(reversed, oracle UNSUPPORTED)={h_rev:.4f} -> {'ACCEPT' if h_rev <= TAU else 'REJECT'}")
print(f"delta={abs(h_rev - h_good):.4f}")

In [ ]:
assert h_good <= TAU and h_rev <= TAU  # verdict preserved
assert abs(h_rev - h_good) < 0.05  # sensor barely notices the binding swap
print("Content attack: verdict preserved despite oracle flip (false acceptance).")

## 2 — Context attack: support deletion flips the verdict

Remove the decisive passage (axis 0) while topical neighbours remain. Sensitivity demands risk increase: the same claim should look less contained.

In [ ]:
E_del = E[1:, :]  # decisive direction deleted, topical rest kept
h_del = hallucination_energy(c_good, E_del, 4)
print(f"H(full evidence)={h_good:.4f} -> {'ACCEPT' if h_good <= TAU else 'REJECT'}")
print(f"H(after deletion)={h_del:.4f} -> {'ACCEPT' if h_del <= TAU else 'REJECT'}")
print(f"delta={h_del - h_good:.4f}")

In [ ]:
assert h_del - h_good > 0.20
assert h_del > TAU >= h_good  # verdict flips to REJECT
print("Context attack: deleting sole support flips ACCEPT to REJECT.")

## 3 — Configuration attack: rank change moves the ruler

Same claim and same evidence; only retained rank changes. A capacity cut redirects the fixed basis, so a supported claim can flip to REJECT without any factual change.

In [ ]:
h_r1 = hallucination_energy(c_good, E, 1)
h_r4 = hallucination_energy(c_good, E, 4)
print(f"H(rank=1)={h_r1:.4f} -> {'ACCEPT' if h_r1 <= TAU else 'REJECT'}")
print(f"H(rank=4)={h_r4:.4f} -> {'ACCEPT' if h_r4 <= TAU else 'REJECT'}")

In [ ]:
assert h_r1 > 0.50 >= h_r4
assert h_r1 > TAU >= h_r4  # same facts, different decision
print("Configuration attack: rank alone flips the verdict.")

## What we earned

The taxonomy predicts behaviour: content (reversal) preserved the verdict, context (support deletion) flipped it toward suspicion, and configuration (rank) flipped it with facts held fixed. Freezing the threshold made each induced flip attributable to the attack, not to retuning.

Chapter 8 asks why this pattern must exist: what does the reduction pipeline discard that no threshold can recover?